# Training, evaluation, and checkpoints

Train a small digit classifier, select an epoch using validation data, and restore a CPU checkpoint with identical predictions. The dataset is the built-in **8×8 scikit-learn digits dataset**, not MNIST.

**Prerequisites:** tensors and an MLP. **Environment:** base requirements plus CPU PyTorch; no torchvision or TensorFlow is needed. **Execution:** restart and run all cells. No data or model download is required; dependency installation requires internet.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device('cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


## Split before fitting preprocessing

Keep the test set untouched until the checkpoint is chosen. Fit the scaler on training data only, and retain its statistics with the model so inference uses the same transformation.


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.2, stratify=y_development, random_state=SEED
)
print('Train / validation / test:', len(y_train), len(y_validation), len(y_test))
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train).astype(np.float32)
X_validation_scaled = scaler.transform(X_validation).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)


## Train on training data and select on validation data

Each batch clears accumulated gradients, computes the loss, backpropagates, and updates parameters. `train()` enables training behavior; `eval()` and `no_grad()` are used for evaluation. The helper below retains a **copy** of the weights with the lowest validation loss. The test set is not used to choose an epoch or model.

Seeds make this small CPU experiment repeatable in the same environment. Bitwise equality across different library versions or hardware is not promised.


In [ ]:
def loader(features, targets, shuffle=False, batch_size=64):
    dataset = TensorDataset(torch.as_tensor(features, dtype=torch.float32),
                            torch.as_tensor(targets, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def evaluate_classifier(model, batches):
    model.eval()  # disables dropout and uses inference behavior
    total_loss, total, correct = 0.0, 0, 0
    all_predictions = []
    with torch.no_grad():  # validation does not build a gradient graph
        for features, targets in batches:
            logits = model(features.to(DEVICE))
            targets = targets.to(DEVICE)
            total_loss += nn.functional.cross_entropy(logits, targets, reduction='sum').item()
            prediction = logits.argmax(dim=1)
            correct += (prediction == targets).sum().item()
            total += targets.numel()
            all_predictions.append(prediction.cpu().numpy())
    return total_loss / total, correct / total, np.concatenate(all_predictions)

def fit_classifier(model, train_batches, validation_batches, epochs=10, learning_rate=0.005):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state, best_validation_loss = [], None, float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total = 0.0, 0
        for features, targets in train_batches:
            features, targets = features.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(features), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * targets.numel()
            total += targets.numel()
        validation_loss, validation_accuracy, _ = evaluate_classifier(model, validation_batches)
        history.append({'epoch': epoch, 'train_loss': total_loss / total,
                        'validation_loss': validation_loss,
                        'validation_accuracy': validation_accuracy})
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history)


In [ ]:
class DigitMLP(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(64, hidden), nn.ReLU(), nn.Dropout(0.1),
                                    nn.Linear(hidden, 10))
    def forward(self, features):
        return self.layers(features)

torch.manual_seed(SEED)
model = DigitMLP().to(DEVICE)
train_batches = loader(X_train_scaled, y_train, shuffle=True)
validation_batches = loader(X_validation_scaled, y_validation)
history = fit_classifier(model, train_batches, validation_batches, epochs=12, learning_rate=0.008)
display(history.tail())
history.plot(x='epoch', y=['train_loss', 'validation_loss'], figsize=(6, 3))
plt.ylabel('Cross-entropy')
plt.show()


## Save tensors and restore with an explicit architecture

The checkpoint contains `state_dict` weights, simple architecture metadata, and preprocessing tensors. Saving the entire model object couples loading to Python classes; this lesson instead constructs the architecture and loads its weights. `map_location='cpu'` supports CPU loading, and `weights_only=True` restricts deserialization. Load only the local checkpoint created here or other trusted artifacts.

An inference checkpoint is sufficient for prediction. Resuming training exactly would additionally require optimizer, scheduler, epoch, and random-generator state.


In [ ]:
import io

checkpoint = {'model_state': model.state_dict(), 'hidden': 48,
              'mean': torch.tensor(scaler.mean_, dtype=torch.float64),
              'scale': torch.tensor(scaler.scale_, dtype=torch.float64), 'seed': SEED}
buffer = io.BytesIO()
torch.save(checkpoint, buffer)
saved_bytes = buffer.tell()
buffer.seek(0)
restored_checkpoint = torch.load(buffer, map_location='cpu', weights_only=True)
restored = DigitMLP(hidden=restored_checkpoint['hidden']).to(DEVICE)
restored.load_state_dict(restored_checkpoint['model_state'])
restored.eval()

# Compare validation logits without consulting the final test set.
probe = torch.tensor(X_validation_scaled[:16])
with torch.no_grad():
    torch.testing.assert_close(model(probe), restored(probe), rtol=0, atol=0)
print('Validation logits match exactly. Checkpoint bytes:', saved_bytes)


## Evaluate the restored model once on the test set

Use the stored preprocessing statistics on raw test inputs. The reported scores describe this split of a small educational dataset and are not evidence of real-world handwriting performance.


In [ ]:
mean = restored_checkpoint['mean'].numpy()
scale = restored_checkpoint['scale'].numpy()
restored_test = ((X_test - mean) / scale).astype(np.float32)
np.testing.assert_allclose(restored_test, X_test_scaled, rtol=1e-5, atol=1e-6)
test_loss, test_accuracy, predictions = evaluate_classifier(restored, loader(restored_test, y_test))
print('Held-out loss:', round(test_loss, 4), 'Accuracy:', round(test_accuracy, 4))
print(classification_report(y_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.show()


## Practice

Compare dropout settings using training/validation data, then evaluate the selected setting on a fresh test split. Save the local checkpoint under the ignored `artifacts/` directory if you want it to persist beyond the kernel.

**Reference:** [PyTorch: saving and loading models](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html).
